# Radix Tree Prefix Caching in vLLM & SGLang

## 1. The Redundant Prefill Tax in LLM Serving
In multi-turn chat applications, Few-Shot prompting, agentic tool-use loops, and multi-document RAG, incoming queries repeatedly share extensive prefix sequences:
- System Prompts ($500 - 2000$ tokens)
- Few-Shot in-context examples ($1000 - 4000$ tokens)
- Long RAG reference PDFs ($10,000 - 30,000$ tokens)

Standard LLM engines re-run the computationally intensive **Prefill Phase** on the entire prompt for every single interaction, wasting up to **$85\%$ of GPU FLOPs** recalculating identical Key-Value tensors!

### Radix Tree (Patricia Trie) Prefix Cache
To solve this, advanced inference runtimes (**vLLM, SGLang, TensorRT-LLM**) store allocated KV cache memory blocks in a **Radix Tree**:
- Edges represent sequences of token IDs.
- Nodes maintain references to allocated physical KV cache memory blocks on the GPU.
- Longest Common Prefix (LCP) matching enables immediate KV cache reuse for any shared prefix, reducing prefill time from seconds to milliseconds!

In [ ]:
class RadixNode:
    def __init__(self, token_seq=None, kv_block_id=None):
        self.token_seq = token_seq or [] # List of token IDs
        self.kv_block_id = kv_block_id   # Physical GPU block pointer
        self.children = {}               # first_token -> RadixNode
        self.last_accessed = 0

class RadixPrefixCache:
    """
    Demonstrates Radix Tree (Patricia Trie) KV Cache indexing and Longest Common Prefix matching.
    """
    def __init__(self):
        self.root = RadixNode()
        self.time_counter = 0

    def match_prefix(self, prompt_tokens):
        """
        Finds the longest cached prefix and returns the matching physical KV blocks.
        """
        curr = self.root
        matched_blocks = []
        tokens_matched = 0
        idx = 0
        
        while idx < len(prompt_tokens):
            first_tok = prompt_tokens[idx]
            if first_tok not in curr.children:
                break
                
            child = curr.children[first_tok]
            seq = child.token_seq
            match_len = 0
            
            # Match token-by-token
            while match_len < len(seq) and (idx + match_len) < len(prompt_tokens):
                if prompt_tokens[idx + match_len] == seq[match_len]:
                    match_len += 1
                else:
                    break
                    
            if match_len == len(seq): # Full node matched
                matched_blocks.append(child.kv_block_id)
                tokens_matched += match_len
                idx += match_len
                curr = child
            else:
                break # Partial match ends traversal
                
        return matched_blocks, tokens_matched

    def insert(self, prompt_tokens, kv_block_id):
        """Simple node insertion for demonstration."""
        if not prompt_tokens:
            return
        first = prompt_tokens[0]
        self.root.children[first] = RadixNode(token_seq=prompt_tokens, kv_block_id=kv_block_id)

# Demonstration
cache = RadixPrefixCache()

# Pre-cache system prompt tokens: [101, 2045, 1037, 3052] -> Physical Block 42
system_prompt_tokens = [101, 2045, 1037, 3052]
cache.insert(system_prompt_tokens, kv_block_id="GPU_BLOCK_42")

# Incoming Request 1: [System Prompt] + "User question A"
request_tokens = [101, 2045, 1037, 3052, 9999, 8888]

matched_blocks, num_matched = cache.match_prefix(request_tokens)
print(f"Request Total Tokens     : {len(request_tokens)}")
print(f"Cached Prefix Tokens Matched : {num_matched} / {len(system_prompt_tokens)}")
print(f"Reused GPU KV Blocks     : {matched_blocks}")
print(f"FLOPs Saved on Prefill   : {(num_matched / len(request_tokens))*100:.1f}% prefill computation avoided!")

## 2. LRU Eviction Under Memory Pressure
When GPU memory pool is full and a new request arrives, the Radix Tree identifies the leaf nodes with the oldest `last_accessed` timestamps, frees their physical KV blocks back to the allocator, and prunes the trie branches.

## 3. High-Yield Interview Questions & Best Practices

### Q1: Why is a Radix Tree superior to a flat Hash Map for prefix caching?
**Answer:** A flat hash map requires an exact string or token hash match. If a user prompt has 1000 tokens of system prompt followed by a new query, hashing the full prompt results in a cache miss. Even hashing token slices requires $O(L)$ hash lookups. A **Radix Tree** naturally represents hierarchical prefixes: finding the longest prefix match requires a single trie traversal proportional to prefix depth, and allows multi-turn conversations to branch seamlessly without duplicate storage.

### Q2: What is the interaction between PagedAttention and Radix Prefix Caching?
**Answer:** They operate symbiotically:
- **PagedAttention** provides the virtual memory block allocator: non-contiguous physical memory pages that can be referenced by pointer.
- **Radix Prefix Caching** maintains the logical index of which physical block IDs contain which token sequences. When an LCP match occurs, the serving engine simply mounts the existing physical block IDs into the new request's Block Table via pointer copy with zero memory replication.